In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV ,TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


data = pd.read_csv("../data/processed_data.csv")

data = pd.get_dummies(data,columns=["Type", "Store", "IsHoliday"],drop_first=True)
data = data.sort_values("Date").reset_index(drop = True )

X = data.drop(["Weekly_Sales","Date"], axis=1)
y = data["Weekly_Sales"]

split_index = int(len(data) * 0.8)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

# XGBoost Model
xg_model = XGBRegressor(objective="reg:squarederror",random_state=0,n_jobs=1)

# Parameters
params = {"n_estimators": [100, 200, 300],"max_depth": [3, 5, 7],"learning_rate": [0.01, 0.05, 0.1],"subsample": [0.8, 1.0],"colsample_bytree": [0.8, 1.0]}
tscv = TimeSeriesSplit(n_splits=3)

# Randomized Search
xgb_search = RandomizedSearchCV(estimator=xg_model,param_distributions=params,n_iter=5,cv=tscv,scoring=None,verbose=2,random_state=0)

# Train
xgb_search.fit(X_train, y_train)

print("Best Parameters:")
print(xgb_search.best_params_)

print("\nBest Score:")
print(xgb_search.best_score_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  15.3s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=   9.4s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  13.0s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  10.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  17.2s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  27.9s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=   3.6s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=   8.3s
[CV] END col

In [ ]:
import joblib
best_xgb = xgb_search.best_estimator_


xgb_pred = best_xgb.predict(X_test)


# Save model
joblib.dump(best_xgb, "../models/best_xgboost.pkl")

print("XGBoost model saved successfully!")
comparison = pd.DataFrame({"Actual_Sales": y_test.values,"Predicted_Sales": xgb_pred})

print(comparison.head(20))

XGBoost model saved successfully!
    Actual_Sales  Predicted_Sales
0        6236.25      6983.271484
1        6179.73      6983.271484
2       24139.31     10314.356445
3        4432.28      5288.740234
4        2256.69      5203.589844
5       80312.67     49128.933594
6       53181.69     42395.289062
7        1013.00      4487.610840
8        6937.30      6090.418457
9        4236.23      5864.088379
10         71.82      3748.176514
11      21338.13     16870.818359
12         20.79      7381.985352
13       1489.71      4073.381592
14         20.76      3828.242432
15       8723.84      7078.932617
16      10188.36      5125.328125
17       3125.00      5003.106934
18        198.25      3789.125244
19        486.00      3789.125244


In [3]:
xgb_mae = mean_absolute_error(y_test, xgb_pred)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

xgb_r2 = r2_score(y_test, xgb_pred)

print("XG RMSE:", xgb_rmse)
print("XG MAE:", xgb_mae)
print("XG R2 Score:", xgb_r2)

XG RMSE: 6932.457602819341
XG MAE: 4472.466469601432
XG R2 Score: 0.8999773581867336
